# Robustness & Perturbation AnalysisHow robust is a model's computation? This module injects noise into weights and activations to identify critical parameters, measure noise propagation, and classify circuits as brittle or robust.

## Why This Matters

Robustness analysis measures how sensitive model behavior is to weight noise, identifying critical parameters whose perturbation has outsized effects. This connects to pruning (which parameters can be removed?) and understanding which weights are load-bearing vs. redundant.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jaximport jax.numpy as jnpimport numpy as npfrom irtk import HookedTransformerConfig, HookedTransformercfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)model = HookedTransformer(cfg)key = jax.random.PRNGKey(42)leaves, treedef = jax.tree.flatten(model)new_leaves = []for leaf in leaves:    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:        key, sk = jax.random.split(key)        new_leaves.append(jax.random.normal(sk, leaf.shape, dtype=leaf.dtype) * 0.1)    else:        new_leaves.append(leaf)model = jax.tree.unflatten(treedef, new_leaves)tokens = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])def metric(logits):    return float(logits[-1, 0])

In [ ]:
from irtk.robustness_perturbation import (    weight_noise_tolerance,    critical_parameter_identification,    activation_noise_propagation,    mode_connectivity_probe,    brittle_vs_robust_circuits,)

## Weight Noise ToleranceHow much noise can the model's weights tolerate before performance degrades?

In [ ]:
result = weight_noise_tolerance(model, tokens, metric, noise_scales=[0.001, 0.01, 0.05, 0.1])print(f"Clean metric: {result['clean_metric']:.6f}")for scale, val, drop in zip(result['noise_scales'], result['metric_values'], result['metric_drops']):    print(f"  noise={scale}: metric={val:.6f}, drop={drop:.6f}")print(f"Tolerance threshold: {result['tolerance_threshold']}")

## Critical Parameter IdentificationWhich weight matrices are most sensitive to perturbation?

In [ ]:
result = critical_parameter_identification(model, tokens, metric)print(f"Most critical: {result['most_critical']}")print(f"Least critical: {result['least_critical']}")print(f"\nSensitivity ranking:")for name, score in result['sensitivity_ranking'][:6]:    print(f"  {name}: {score:.6f}")

## Activation Noise PropagationHow does noise amplify or dampen as it passes through layers?

In [ ]:
result = activation_noise_propagation(model, tokens)for l in range(len(result['output_perturbations'])):    print(f"  Layer {l}: perturbation={result['output_perturbations'][l]:.6f}, "          f"amplification={result['amplification_factors'][l]:.4f}x")print(f"Most amplifying: layer {result['most_amplifying_layer']}")print(f"Noise stability: {result['noise_stability']:.4f}")

## Mode ConnectivityIs the loss landscape smooth between different weight configurations?

In [ ]:
model_b = jax.tree.unflatten(treedef, [    jax.random.normal(jax.random.PRNGKey(99), l.shape, dtype=l.dtype) * 0.1    if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l    for l in jax.tree.leaves(model)])result = mode_connectivity_probe(model, model_b, tokens, metric)print(f"Connected: {result['is_connected']}")print(f"Smoothness: {result['smoothness']:.4f}")print(f"Max barrier: {result['max_barrier']:.6f}")

## Brittle vs Robust HeadsWhich attention heads fail gracefully under noise vs catastrophically?

In [ ]:
result = brittle_vs_robust_circuits(model, tokens, metric)print(f"Mean robustness: {result['mean_robustness']:.4f}")print(f"Most brittle: L{result['most_brittle'][0]}H{result['most_brittle'][1]}")print(f"Brittle heads: {result['brittle_heads']}")print(f"Robust heads: {result['robust_heads']}")